# 澳门与香港日度访客量：数据管道

论文任务：27 条澳门客源地序列 + 2 条香港访客类别序列；每个起点只使用已知历史，预测未来 28 天。源数据只读，零值保留。请从本 notebook 所在文件夹运行。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(value): print(value.to_string() if hasattr(value, 'to_string') else value)
from data_pipeline import (
    MACAU_DIR, HONG_KONG_WORKBOOK, TEST_START, TEST_END,
    load_all, quality_summary, rolling_origins, make_input_task,
    write_data_snapshot, write_task_artifacts,
)
PROJECT = Path.cwd()
assert (PROJECT / 'data_pipeline.py').exists(), '请将工作目录设为 paper_pipeline 文件夹'
print('Macau:', MACAU_DIR)
print('Hong Kong workbook:', HONG_KONG_WORKBOOK)
print('Target-date test period:', TEST_START, 'to', TEST_END)


## 1. 读取与核对

澳门只读 `data,Tourism` CSV。香港只读工作簿的 `每日数据`：内地访客和其他访客分别为两条目标序列；工作簿中的访客总量仅用于对账。

In [ ]:
series = load_all()
qa = pd.DataFrame(quality_summary(series))
display(qa.groupby('destination').agg(series=('market','count'),
                                      first_date=('first_date','min'),
                                      last_date=('last_date','max'),
                                      total_days=('days','sum'),
                                      zero_days=('zero_days','sum')))
display(qa.head(4))
assert len(series) == 29
assert (qa['days'] > 0).all()


## 2. 生成标准化快照、模型输入与隔离目标

`task_inputs.parquet` 不含真实目标；`targets.parquet` 只允许独立评估器读取。manifest 仅保留逻辑来源标识和 SHA256，不保存本机绝对路径。

In [ ]:
manifest = write_data_snapshot(PROJECT / 'data')
task_manifest = write_task_artifacts(series, PROJECT / 'data')
print('series:', manifest['series_count'])
print('normalized observations:', sum(item['days'] for item in manifest['quality']))
print('source files hashed:', len(manifest['source_sha256']))
print('isolated tasks:', task_manifest['task_count'])


## 3. 事先固定滚动起点

完整样本每日滚动（stride=1）；共同计算样本从 2023-12-31 起每隔 28 天滚动（stride=28）。每条未来 28 天路径都完整落在 2024-01-01—2026-07-31。

In [ ]:
for key in ['Macau:CN_Chinese_mainland', 'HongKong:Mainland visitors', 'HongKong:Other visitors']:
    item = series[key]
    full = rolling_origins(item, stride_days=1)
    common = rolling_origins(item, stride_days=28)
    print(key, '| full:', len(full), full[0], 'to', full[-1],
          '| common:', len(common), common[0], 'to', common[-1])


## 4. 检查模型输入的信息边界

`task.history` 截止 2026-07-03，任务对象没有 `actual` 属性。真实目标只存在于单独的 `targets.parquet`。

In [ ]:
from datetime import date
item = series['Macau:CN_Chinese_mainland']
task = make_input_task(item, date(2026, 7, 3), history_length=180)
print(task.key, '| history:', task.history_start, 'to', task.origin, len(task.history),
      '| forecast dates:', task.target_dates[0], 'to', task.target_dates[-1])
assert len(task.history) == 180 and len(task.target_dates) == 28
assert not hasattr(task, 'actual')
assert task.target_dates[0] > task.origin
assert task.target_dates[-1] <= TEST_END
assert np.array_equal(task.history, item.values.loc[str(task.history_start):str(task.origin)].to_numpy())


下一步运行 `02_统一模型与回测.ipynb`。本 notebook 只做数据结构与时间边界检查，不训练模型。